<a href="https://colab.research.google.com/github/hasby-umutoniwabo/TimeSeriesForecasting-formative/blob/main/TimeSeries_formative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q kagglehub

Adding kaggle credentials via colab secrets

In [3]:
from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("Kaggle credentials loaded from Colab Secrets.")

Kaggle credentials loaded from Colab Secrets.


Download the dataset

In [13]:
import kagglehub
import os

dataset_path = kagglehub.dataset_download("freckled/telecom")
file1_path = os.path.join(dataset_path, "data1.csv")
file2_path = os.path.join(dataset_path, "data2.csv")

print("dataset_path:", dataset_path)
print("file1_path:", file1_path)
print("file2_path:", file2_path)

Using Colab cache for faster access to the 'telecom' dataset.
dataset_path: /kaggle/input/telecom
file1_path: /kaggle/input/telecom/data1.csv
file2_path: /kaggle/input/telecom/data2.csv


Inspect what we got

In [14]:
import os

# List every file in the dataset folder with its size in MB
for fname in sorted(os.listdir(dataset_path)):
    full_path = os.path.join(dataset_path, fname)
    size_mb = os.path.getsize(full_path) / 1e6
    print(f"{fname:45s} {size_mb:8.1f} MB")

data1.csv                                      10318.7 MB
data2.csv                                      10419.5 MB


Peek at the raw structure of data1.csv

In [15]:
import os

# full path to the first file inside the downloaded dataset folder
file1_path = os.path.join(dataset_path, "data1.csv")

# Read just the first 5 lines (no much memory cause the whole thing isn't being parsed yet)
with open(file1_path, "r") as f:
    for i in range(5):
        line = f.readline()
        print(f"Line {i}: {line}")

Line 0: GridID,TimeInterval,countrycode,smsin,smsout,callin,callout,internet

Line 1: 1,1383260400000,0,0.0813626235112588,,,,

Line 2: 1,1383260400000,39,0.1418642547024292,0.1567870050390246,0.1609379369170182,0.0522748485285732,11.028366381681026

Line 3: 1,1383261000000,0,0.136587822758231,,,0.0273004648771861,

Line 4: 1,1383261000000,33,,,,,0.0261374242642866



Count total rows without loading the file into memory

In [16]:
# Counting lines by streaming through the file in chunks (avoids ever holding the full 10GB file in RAM at once)
def count_lines(path, chunk_size=1024*1024*10):  # read 10MB at a time
    count = 0
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            count += chunk.count(b"\n")
    return count

num_lines = count_lines(file1_path)
print(f"data1.csv has approximately {num_lines:,} lines")

data1.csv has approximately 160,108,004 lines


Install a memory-measuring tool

In [17]:
#psutil lets us check how much RAM our own Colab process is using, so we can prove the "before" vs "after" memory improvement with real numbers
!pip install -q psutil

BASELINE: naive loading (all columns, default types)

In [18]:
import psutil
import os
import pandas as pd

def memory_used_mb():
    # Returns how much RAM the current Python process is using, in MB
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1e6

mem_before = memory_used_mb()
print(f"Memory before loading: {mem_before:.1f} MB")

# Naive approach: load a sample of rows with ALL columns, letting pandas guess types.
sample_naive = pd.read_csv(file1_path, nrows=2_000_000)

mem_after = memory_used_mb()
print(f"Memory after loading 2,000,000 rows (naive): {mem_after:.1f} MB")
print(f"Memory used by this data: {mem_after - mem_before:.1f} MB")
print(f"Data types pandas chose:\n{sample_naive.dtypes}")

# Clean up before the next test, so measurements don't overlap
del sample_naive

Memory before loading: 1519.4 MB
Memory after loading 2,000,000 rows (naive): 1647.4 MB
Memory used by this data: 128.0 MB
Data types pandas chose:
GridID            int64
TimeInterval      int64
countrycode       int64
smsin           float64
smsout          float64
callin          float64
callout         float64
internet        float64
dtype: object


OPTIMIZED: Only the columns we need, efficient dtypes

In [19]:
mem_before2 = memory_used_mb()
print(f"Memory before loading: {mem_before2:.1f} MB")
sample_optimized = pd.read_csv(
    file1_path,
    nrows=2_000_000,
    usecols=["GridID", "TimeInterval", "internet"],
    dtype={"GridID": "int16", "TimeInterval": "int64", "internet": "float32"},
)

mem_after2 = memory_used_mb()
print(f"Memory after loading 2,000,000 rows (optimized): {mem_after2:.1f} MB")
print(f"Data used by this data: {mem_after2 - mem_before2:.1f} MB")
print(f"Data types used:\n{sample_optimized.dtypes}")

del sample_optimized

Memory before loading: 1519.4 MB
Memory after loading 2,000,000 rows (optimized): 1563.4 MB
Data used by this data: 44.0 MB
Data types used:
GridID            int16
TimeInterval      int64
internet        float32
dtype: object


Find the time range so we can size our array

In [20]:
# we just need min/max timestamp across both files, so read only that one column
t1 = pd.read_csv(file1_path, usecols=["TimeInterval"], dtype={"TimeInterval": "int64"})
t2 = pd.read_csv(file2_path, usecols=["TimeInterval"], dtype={"TimeInterval": "int64"})

t_min = min(t1["TimeInterval"].min(), t2["TimeInterval"].min())
t_max = max(t1["TimeInterval"].max(), t2["TimeInterval"].max())

del t1, t2  # done with these, free the memory

n_slots = (t_max - t_min) // 600_000 + 1  # 600,000 ms = 10 minutes
print("time range:", t_min, "to", t_max)
print("number of 10-min slots:", n_slots)

time range: 1383260400000 to 1388616600000
number of 10-min slots: 8928


Set up one small array to hold everything



In [21]:
import numpy as np

n_grids = 10000
traffic = np.zeros((n_slots, n_grids), dtype="float32")
print("array size in memory (MB):", traffic.nbytes / 1e6)

array size in memory (MB): 357.12


Stream through a file and accumulate straight into the array

In [23]:
def accumulate_file(path, t_min, traffic):
    for chunk in pd.read_csv(
        path,
        usecols=["GridID", "TimeInterval", "internet"],
        dtype={"GridID": "int16", "TimeInterval": "int64", "internet": "float32"},
        chunksize=5_000_000,
    ):
        chunk = chunk.dropna(subset=["internet"])  # rows with no internet activity, skip them

        row_idx = ((chunk["TimeInterval"].values - t_min) // 600_000).astype("int64")
        col_idx = (chunk["GridID"].values - 1).astype("int64")  # grid ids start at 1

        # np.add.at handles repeated (row, col) pairs correctly, unlike normal indexing
        np.add.at(traffic, (row_idx, col_idx), chunk["internet"].values)

Accumulate both files into the array

In [24]:
mem_before = memory_used_mb()

accumulate_file(file1_path, t_min, traffic)
print("file 1 done")

accumulate_file(file2_path, t_min, traffic)
print("file 2 done")

mem_after = memory_used_mb()
print(f"memory used: {mem_after - mem_before:.1f} MB")
print("total traffic recorded:", traffic.sum())

file 1 done
file 2 done
memory used: 438.8 MB
total traffic recorded: 5552899600.0


In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Proper time index to save everything

In [27]:
import pandas as pd
import numpy as np

# turn the raw slot numbers back into real datetimes, one per row of our array
time_index = pd.to_datetime(t_min, unit="ms") + pd.to_timedelta(np.arange(n_slots) * 10, unit="min")

# make a folder in Drive for this project so things stay organized
save_dir = "/content/drive/MyDrive/milan_traffic_project"
os.makedirs(save_dir, exist_ok=True)

# save as .npz — compact binary format, keeps the array + timestamps together
np.savez_compressed(
    os.path.join(save_dir, "traffic_matrix.npz"),
    traffic=traffic,
    timestamps=time_index.values,
)

print("saved to:", os.path.join(save_dir, "traffic_matrix.npz"))
print("file size (MB):", os.path.getsize(os.path.join(save_dir, "traffic_matrix.npz")) / 1e6)

saved to: /content/drive/MyDrive/milan_traffic_project/traffic_matrix.npz
file size (MB): 295.918121


Sanity check: loading it back and confirm it matches

In [28]:
loaded = np.load(os.path.join(save_dir, "traffic_matrix.npz"), allow_pickle=True)
loaded_traffic = loaded["traffic"]
loaded_timestamps = loaded["timestamps"]

print("shape:", loaded_traffic.shape)
print("first timestamp:", loaded_timestamps[0])
print("last timestamp:", loaded_timestamps[-1])
print("total traffic matches:", np.isclose(loaded_traffic.sum(), traffic.sum()))

shape: (8928, 10000)
first timestamp: 2013-10-31T23:00:00.000000000
last timestamp: 2014-01-01T22:50:00.000000000
total traffic matches: True
